## Lab Measurements

@TODO: Impute with rate over time or decay... timeseries features from kreatinine and eGFR (CKD-EPI)

In [ ]:
import pandas            as pd
import miceforest        as mf
import matplotlib.pyplot as plt
import numpy             as np
import os
import missingno

In [ ]:
os.chdir(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805')

In [ ]:
df_lab = pd.read_sas('lab_20240805.sas7bdat')

In [ ]:
str_columns = df_lab.select_dtypes(object).columns
for c in str_columns:
    df_lab[c] = df_lab[c].str.decode(encoding='latin1')

In [ ]:
df_lab.studyId_0831 = df_lab.studyId_0831.astype(int) # Converto to int to have more clear view

In [ ]:
df_lab.columns # How many of those are relevant to us?

Let's remove uneuseful columns

In [ ]:
df_lab = df_lab.drop(['identifier_value', 'basedOn_ProcedureRequest_value',
                      'LabID', 'category3_display_original', 'category4_code_original',
                      'code_code_original','code2_code_original',
                      'code2_display_original', 'code3_display_original',
                      'interpretation_display', 'referenceRange_text',
                      'code3_system_original', 'code4_display_original', 'comment',
                      'status_display_original', 'valueQuantity_code_original',
                      'valueQuantity_comparator', 'valueQuantity_unit',
                      'valueQuantity_unit_original', 'valueString',
                      'Practitioner_Rol', 'gender', 'organization_Organization_value',
                      'specialty_specialtyAGB_display', 'index_date'],axis='columns')

The most recurrent row is patient `12482107839` that appears in this table _914 times_...<br>
Let's make this smart to convert the table in wide format

In [ ]:
reps = df_lab.studyId_0831.value_counts().to_list()


In [ ]:
id = []
unique_id = df_lab.studyId_0831.unique()
for subj,rep in enumerate(reps):
    tmp = np.repeat(unique_id[subj], rep)
    id.append(tmp)
    

In [ ]:
subjs = pd.Series(np.concat(id))
subjs

Impute values based on some nice MICE Forest algorithm:
- Pivot on <code>code_display_original</code>
- Run MICE Forest

In [ ]:
df_lab_pivoted = df_lab.pivot(columns='code_display_original', values='valueQuantity_value')
df_lab_pivoted

In [ ]:
df_lab_pivoted['eGFR (CKD-EPI)'].plot()

In [ ]:
missingno.matrix(df_lab_pivoted)

In [ ]:
ker = mf.ImputationKernel(
    df_lab_pivoted.reset_index(drop=True),
    num_datasets=2,
    save_all_iterations_data=True,
    random_state=42)

ker.mice(iterations = 5,
         n_estimators = 100,
         boosting = 'gbdt',
         min_sum_hessian_in_leaf = 0.01)


In [ ]:

df_lab_imputed = ker.complete_data()


In [ ]:
df_lab_imputed['studyId_0831'] = subjs

In [ ]:
df_lab_imputed.Kreatinine.plot()

In [ ]:
missingno.matrix(df_lab_imputed)

In [ ]:
#ker.plot_imputed_distributions()

Try to aggregate values with using mean

In [ ]:
def wavg_last_n(x : pd.Series):
    weights = np.arange(start=1,stop=x.count()+1)/x.count()
    return np.average(x,weights=weights)

In [ ]:
df_lab_imputed.reset_index(drop=True)\
                .groupby('studyId_0831')\
                    .agg(lambda x: wavg_last_n(x))

In [ ]:
df_lab_imputed['CRP'].plot(kind='box')

In [ ]:
df_lab_imputed.reset_index(drop=True)\
        .groupby('studyId_0831')\
            .agg(lambda x: wavg_last_n(x))\
                .to_csv(r'C:\Users\jvitale\Desktop\Odin\ODIN\sandboxes\02_lab_meas\lab_meas_imputed.csv')